# 2 · Entrenar y publicar los modelos

Vamos a entrenar **dos modelos** con los datos de funcionamiento **normal**:

* **A. Curva de potencia** (regresión con una **red neuronal**): dado el viento, ¿cuánto *debería* producir?
  Si produce mucho menos, algo pasa (aunque la máquina no avise).
* **B. Detector de anomalías** (*Isolation Forest*): ¿esta combinación de viento, potencia, vibración y
  temperatura es habitual?

Se entrena **solo con lo normal** porque es lo que abunda en una planta real: de las averías hay pocos ejemplos.

In [ ]:
from labia import datos, guardar
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

df = datos("aerolab")

# Segundos desde el último cambio de escenario: justo después de un cambio la máquina todavía
# está llegando a su nuevo estado (la góndola gira, el rotor acelera, la temperatura sube).
tramo = (df.escenario != df.escenario.shift()).cumsum()
df["seg_desde_cambio"] = (df.ts - df.groupby(tramo).ts.transform("min")).dt.total_seconds()

TRANSICION = 60
normal = df[(df.escenario == "normal") & (df.averia == 0) & (df.seg_desde_cambio >= TRANSICION)]
print(len(normal), "filas normales de", len(df), f"(sin los {TRANSICION} s posteriores a cada cambio)")

> **Etiqueta frente a realidad.** La etiqueta cambia al pulsar el botón, pero la máquina tarda en reaccionar.
> Si entrenas con esas filas de transición, el modelo aprende como «normal» algo que no lo es. Prueba a poner
> `TRANSICION = 0` y compara los resultados.

## A · Curva de potencia

Separamos un 25 % de los datos para **evaluar** con filas que el modelo no ha visto al entrenar.
Comparamos una recta (regresión lineal) con una red neuronal pequeña (dos capas de 32 neuronas).

In [ ]:
X = normal[["viento"]]
y = normal["potencia"]
X_ent, X_prueba, y_ent, y_prueba = train_test_split(X, y, test_size=0.25, random_state=0)

from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

recta = LinearRegression().fit(X_ent, y_ent)
red = make_pipeline(StandardScaler(),
                    MLPRegressor(hidden_layer_sizes=(32, 32), max_iter=3000, random_state=0)).fit(X_ent, y_ent)

for nombre, m in [("recta", recta), ("red neuronal", red)]:
    p = m.predict(X_prueba)
    print(f"{nombre:13s} MAE = {mean_absolute_error(y_prueba, p):6.0f} kW   R² = {r2_score(y_prueba, p):.3f}")

In [ ]:
v = pd.DataFrame({"viento": np.linspace(0, 20, 200)})
plt.figure(figsize=(9, 5))
plt.scatter(normal.viento, normal.potencia, s=5, alpha=.3, label="datos normales")
plt.plot(v.viento, recta.predict(v), "g--", label="recta")
plt.plot(v.viento, red.predict(v), "r", lw=2, label="red neuronal")
plt.xlabel("viento (m/s)"); plt.ylabel("potencia (kW)"); plt.legend(); plt.grid(alpha=.3);

### ¿Sirve para detectar problemas?

Aplicamos la red a **todos** los datos y miramos la **desviación** = potencia real − potencia esperada.

In [ ]:
df["esperada"] = red.predict(df[["viento"]])
df["desviacion"] = df.potencia - df.esperada
df[df.seg_desde_cambio >= TRANSICION].boxplot(column="desviacion", by="escenario", figsize=(9, 4), rot=15)
plt.suptitle(""); plt.title("Desviación respecto a la curva aprendida (kW)"); plt.ylabel("kW");

Si te convence, **publícalo**. El nombre `aerolab-curva` es el que usará el nodo **ia predecir** en Node-RED,
y las *entradas* son las claves que tendrá que traer el mensaje.

In [ ]:
mae = float(mean_absolute_error(y_prueba, red.predict(X_prueba)))
guardar(red, "aerolab-curva", entradas=["viento"], tipo="regresion", salida="potencia",
        descripcion="Curva de potencia aprendida en funcionamiento normal", metricas={"mae_kw": round(mae, 1)})

## B · Detector de anomalías

El *Isolation Forest* aprende qué combinaciones de valores son habituales. `contamination=0.01` le dice que
considere raro el 1 % más extraño de lo que ha visto (el umbral).

In [ ]:
from sklearn.ensemble import IsolationForest

ENTRADAS = ["viento", "potencia", "vibracion", "temp_rodamiento"]
iso = make_pipeline(StandardScaler(), IsolationForest(contamination=0.01, random_state=0))
iso.fit(normal[ENTRADAS])

df["anomalia"] = iso.predict(df[ENTRADAS]) == -1      # -1 = anómalo
resumen = df[df.seg_desde_cambio >= TRANSICION].groupby("escenario")["anomalia"].mean().mul(100).round(1)
resumen.to_frame("% de filas marcadas como anómalas")

### Evaluar como un detector

* **Falsas alarmas**: filas normales marcadas como anómalas.
* **Detección**: filas no normales marcadas como anómalas.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

ev = df[df.seg_desde_cambio >= TRANSICION]          # evaluamos fuera de las transiciones
real = ev.escenario != "normal"
ConfusionMatrixDisplay(confusion_matrix(real, ev.anomalia), display_labels=["normal", "no normal"]).plot()
plt.xlabel("según el modelo"); plt.ylabel("realidad");
print(f"falsas alarmas: {100 * ev.anomalia[~real].mean():.1f} %   detección: {100 * ev.anomalia[real].mean():.1f} %")

In [ ]:
guardar(iso, "aerolab-anomalias", entradas=ENTRADAS, tipo="anomalias",
        descripcion="Isolation Forest entrenado con funcionamiento normal",
        metricas={"falsas_alarmas_pct": round(float(100 * ev.anomalia[~real].mean()), 1),
                  "deteccion_pct": round(float(100 * ev.anomalia[real].mean()), 1)})

## ✏️ Tu turno

1. Añade `rotor_rpm` o `pitch` a las entradas de la **curva de potencia**. ¿Mejora el MAE? Ojo: en Node-RED
   tendrás que enviar también esas claves.
2. Cambia `contamination` a 0.05 y a 0.001. ¿Qué pasa con las falsas alarmas y con la detección?
3. Quita `potencia` de las entradas del detector. ¿Sigue viendo la góndola desalineada? ¿Por qué?
4. Pon `TRANSICION = 0`. ¿Cuánto empeoran el MAE y la detección? ¿Por qué? (pista: la góndola tarda en girar
   y la temperatura en subir).

Cada vez que vuelvas a `guardar(...)` con el mismo nombre se crea una **versión nueva**; Node-RED la usa al momento.